## Goal of this notebook

Many real-world LLM applications require multiple steps of reasoning or processing.

Instead of using a single prompt, we often need to:

1. Generate intermediate results
2. Pass those results to another prompt
3. Combine multiple outputs into a final response

LangChain provides **Chains** to build these multi-step pipelines.

In this notebook we will explore four types of chains:

- **LLMChain** – a single prompt → model → output
- **SimpleSequentialChain** – chaining outputs linearly
- **SequentialChain** – managing multiple inputs and outputs
- **RouterChain** – dynamically selecting the best chain

### Chain Architecture

A typical LLM pipeline looks like this:

User Input  
      ↓  
Prompt Template  
      ↓  
LLM  
      ↓  
Intermediate Output  
      ↓  
Next Prompt  
      ↓  
Final Output

In [1]:
# The updated notebook uses current LangChain APIs,

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

## Environment Setup

We start by loading environment variables and importing the required libraries.

The OpenAI API key is loaded using `dotenv`.

We will use:

- LangChain
- OpenAI chat models
- Pandas for working with the dataset

In [3]:
# Use a current chat model directly instead of older dated model names.
llm_model = "gpt-4o-mini"

### Loading Example Data

In this notebook we use a dataset of product reviews.

The reviews will be used as input for different chains to demonstrate how LLM pipelines can process and transform text.

Each chain will perform a different task such as:

- generating product names
- summarizing reviews
- translating text
- generating follow-up responses

In [5]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [6]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

The **LLMChain** is the simplest type of chain.

It consists of three main components:

Prompt Template → Model → Output Parser

The prompt template defines the structure of the prompt,  
the model generates the response,  
and the output parser converts the result into a usable format.

This is the basic building block for more complex chains.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [8]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [9]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [10]:
chain = prompt | llm | StrOutputParser()

In [11]:
product = "Queen Size Sheet Set"
chain.invoke({"product": product})

'Regal Linens Co.'

## Simple Sequential Chain

A **SimpleSequentialChain** allows us to connect multiple LLM steps.

The output of one step becomes the input to the next step.

Example pipeline:

Product Name → Generate Company Name → Generate Description

This approach is useful when tasks must be executed **in a strict sequence**.

In [12]:
from langchain_core.runnables import RunnableLambda

In [13]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = first_prompt | llm | StrOutputParser()

In [14]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = second_prompt | llm | StrOutputParser()

In [15]:
overall_simple_chain = (
    chain_one
    | RunnableLambda(lambda company_name: {"company_name": company_name})
    | chain_two
)

In [16]:
overall_simple_chain.invoke({"product": product})



> Entering new SimpleSequentialChain chain...
Regal Linens Co.
Regal Linens Co. offers luxurious and high-quality linens for bedrooms and bathrooms, providing elegance and comfort for your home.

> Finished chain.


'Regal Linens Co. offers luxurious and high-quality linens for bedrooms and bathrooms, providing elegance and comfort for your home.'

## Sequential Chain

A **SequentialChain** is more flexible than SimpleSequentialChain.

It allows multiple inputs and outputs between steps.

For example, we can build a pipeline that:

1. Translates a review
2. Summarizes the review
3. Detects the language
4. Generates a follow-up response

Each step can use outputs from previous steps.

In [17]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [18]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = (
    RunnableLambda(lambda x: {"Review": x["Review"]})
    | first_prompt
    | llm
    | StrOutputParser()
)


In [19]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = (
    RunnableLambda(lambda x: {"English_Review": x["English_Review"]})
    | second_prompt
    | llm
    | StrOutputParser()
)


In [20]:
# prompt template 3: detect the language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = (
    RunnableLambda(lambda x: {"Review": x["Review"]})
    | third_prompt
    | llm
    | StrOutputParser()
)


In [21]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = (
    RunnableLambda(lambda x: {"summary": x["summary"], "language": x["language"]})
    | fourth_prompt
    | llm
    | StrOutputParser()
)


In [22]:
# overall_chain: input= Review
# and output= English_Review, summary, followup_message
overall_chain = (
    RunnablePassthrough()
    .assign(
        English_Review=chain_one,
        language=chain_three,
    )
    .assign(summary=chain_two)
    .assign(followup_message=chain_four)
    | RunnableLambda(
        lambda x: {
            "Review": x["Review"],
            "English_Review": x["English_Review"],
            "summary": x["summary"],
            "followup_message": x["followup_message"],
        }
    )
)

In [23]:
review = df.Review[5]
overall_chain.invoke({"Review": review})



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': '"I find the taste mediocre. The foam doesn\'t last, it\'s strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?"',
 'summary': 'The reviewer found the taste of the product to be mediocre, with the foam not lasting, leading them to suspect the product may be an old batch or counterfeit.',
 'followup_message': "Merci pour votre avis honnête sur le produit. Il est regrettable d'apprendre que la saveur n'était que moyenne et que la mousse ne durait pas. Votre soupçon selon lequel il pourrait s'agir d'un lot ancien ou contrefait est compréhensible. Il est important de toujours acheter des produits authentiques pour garantir une expérience de qualité. Peut-être pourriez-vous contacter le fabricant pour obtenir plus d'informations sur la provenance du pro

## Router Chain

Sometimes we want to route a request to different prompts depending on the topic.

A **Router Chain** analyzes the input and decides which specialized chain should handle the request.

For example:

Physics question → Physics chain  
Math question → Math chain  
Biology question → Biology chain

This technique allows us to build **multi-expert LLM systems** where each prompt specializes in a specific domain.

In [24]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [25]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "history", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

### Why Router Chains Are Useful

Router chains enable modular LLM systems.

Instead of relying on a single large prompt, we can create specialized prompts for different tasks.

Benefits:

- better accuracy
- clearer prompts
- easier system design

In [26]:
import json

from langchain_core.runnables import RunnableLambda

In [27]:
llm = ChatOpenAI(temperature=0, model=llm_model)

In [28]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = prompt | llm | StrOutputParser()
    destination_chains[name] = chain


destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [29]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

In [30]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model, select the prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what each prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

Return a JSON object with this shape:
{{
    "destination": string,
    "next_inputs": string
}}

REMEMBER: The value of "destination" MUST match one of \
the candidate prompts listed below.\
If "destination" does not fit any of the specified prompts, set it to "DEFAULT".
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}"""

In [31]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = ChatPromptTemplate.from_template(router_template)


def parse_route(response):
    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    route = json.loads(response)
    destination = route["destination"].strip()
    if destination not in destination_chains:
        destination = "DEFAULT"

    return {
        "destination": destination,
        "next_inputs": route["next_inputs"],
    }


router_chain = router_prompt | llm | StrOutputParser() | RunnableLambda(parse_route)

In [32]:
def select_chain(route):
    destination = route["destination"]
    next_inputs = route["next_inputs"]

    print(f"Selected destination: {destination}")

    if destination == "DEFAULT":
        return default_chain.invoke({"input": next_inputs})

    return destination_chains[destination].invoke({"input": next_inputs})


chain = (
    RunnableLambda(lambda user_input: {"input": user_input})
    | router_chain
    | RunnableLambda(select_chain)
)

In [33]:
chain.invoke("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [34]:
chain.invoke("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [35]:
chain.invoke("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell function and structure. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that the genetic information is preserved and can be used to carry out the necessary processes for life.'

## Conclusion

In this notebook we explored different ways to build multi-step LLM workflows using LangChain chains.

Key takeaways:

- **LLMChain** is the basic building block for interacting with a model.
- **SimpleSequentialChain** allows chaining tasks linearly.
- **SequentialChain** supports multiple inputs and outputs between steps.
- **RouterChain** dynamically selects the most appropriate prompt or chain.

These chaining techniques allow developers to build complex LLM pipelines such as assistants, agents, and automated workflows.